In [10]:
import os
import certifi
from gen_ai_hub.proxy.langchain import init_llm
import pandas as pd
from hana_ml import dataframe
from hana_ai.tools.toolkit import HANAMLToolkit

# Ensure httpx/ssl can find a valid CA bundle when SSL_CERT_FILE is set.
os.environ["SSL_CERT_FILE"] = certifi.where()

trust_store = certifi.where()
connection_context = dataframe.ConnectionContext(userkey="RaysKey", encrypt=True,
                                                sslValidateCertificate=True,
                                                sslTrustStore=trust_store)

llm = init_llm('gpt-4o', temperature=0.1, max_tokens=1800)
toolkit = HANAMLToolkit(connection_context, used_tools='all', return_direct={"fetch_data": False})

In [ ]:
# Start MCP server explicitly using SSE transport.
# Note: if the port is occupied, toolkit will automatically try the next port.
toolkit.launch_mcp_server(server_name="test", host="127.0.0.1", transport="sse", port=12345)

# Capture the actual host/port selected by the server (after auto-retry).
from hana_ai.tools.toolkit import HANAMLToolkit

_servers = list(getattr(HANAMLToolkit, "_global_mcp_servers", {}).values())
if not _servers:
    raise RuntimeError("No MCP server found in registry; server may not have started.")

_last = _servers[-1]
MCP_HOST = _last.get("host", "127.0.0.1")
MCP_PORT = int(_last.get("port", 12345))
print(f"✅ MCP server is running at http://{MCP_HOST}:{MCP_PORT}/sse")

✅ MCP server is running at http://127.0.0.1:12347/sse


INFO:     Started server process [20750]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


INFO:     Uvicorn running on http://127.0.0.1:12347 (Press CTRL+C to quit)


INFO:     127.0.0.1:57235 - "GET /sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:57236 - "POST /messages/?session_id=2d58b924479a4dafbbbc1888324b55e9 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57236 - "POST /messages/?session_id=2d58b924479a4dafbbbc1888324b55e9 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57236 - "POST /messages/?session_id=2d58b924479a4dafbbbc1888324b55e9 HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57236 - "POST /messages/?session_id=2d58b924479a4dafbbbc1888324b55e9 HTTP/1.1" 202 Accepted


In [15]:
import os
import traceback
from typing import Any

import anyio
import httpx
from mcp.client.sse import sse_client
from mcp import ClientSession

# Make sure localhost requests do NOT go through corporate proxies.
os.environ.setdefault("NO_PROXY", "127.0.0.1,localhost")
os.environ.setdefault("no_proxy", "127.0.0.1,localhost")


def _httpx_client_no_env(
    headers: dict[str, str] | None = None,
    timeout: httpx.Timeout | None = None,
    auth: httpx.Auth | None = None,
) -> httpx.AsyncClient:
    kwargs: dict[str, Any] = {
        "follow_redirects": True,
        "trust_env": False,
        "timeout": timeout or httpx.Timeout(30.0, read=300.0),
    }
    if headers is not None:
        kwargs["headers"] = headers
    if auth is not None:
        kwargs["auth"] = auth
    return httpx.AsyncClient(**kwargs)


def _print_exception_group(eg: BaseException, prefix: str = "") -> None:
    """Pretty-print ExceptionGroup/TaskGroup errors."""
    if hasattr(eg, "exceptions"):
        exceptions = getattr(eg, "exceptions")
        print(f"{prefix}{type(eg).__name__}: {eg}")
        for idx, sub in enumerate(exceptions):
            _print_exception_group(sub, prefix=f"{prefix}  [{idx}] ")
    else:
        print(f"{prefix}{type(eg).__name__}: {eg}")
        traceback.print_exception(type(eg), eg, eg.__traceback__)


async def get_mcp_tools(host: str, port: int):
    """Get MCP tools using SSE client."""
    url = f"http://{host}:{port}/sse"
    print(f"🔌 Connecting to {url}")

    # Give the background server a moment to finish startup.
    await anyio.sleep(0.8)

    try:
        async with sse_client(
            url,
            httpx_client_factory=_httpx_client_no_env,
            timeout=10,
        ) as streams:
            async with ClientSession(*streams) as session:
                await session.initialize()

                tools = await session.list_tools()
                print(f"✅ successfully obtains {len(tools.tools)} tools:")
                for tool in tools.tools:
                    print(f"  name: {tool.name}")

                # Call the fetch_data tool.
                print("🔍 calling fetch_data...")
                result = await session.call_tool(
                    name="fetch_data",
                    arguments={
                        "table_name": "PAL_COVID_DATA_TBL",
                        "top_n": 5,
                    },
                )
                return result

    except ExceptionGroup as eg:
        print("❌ Failed (ExceptionGroup):")
        _print_exception_group(eg)
        return None
    except Exception as e:
        print("❌ Failed:")
        _print_exception_group(e)
        return None


# Use MCP_HOST / MCP_PORT produced by Cell 2
result = await get_mcp_tools(MCP_HOST, MCP_PORT)

🔌 Connecting to http://127.0.0.1:12347/sse
✅ successfully obtains 26 tools:
  name: accuracy_measure
  name: additive_model_forecast_fit_and_save
  name: additive_model_forecast_load_model_and_predict
  name: automatic_timeseries_fit_and_save
  name: automatic_timeseries_load_model_and_predict
  name: automatic_timeseries_load_model_and_score
  name: cap_artifacts
  name: delete_models
  name: fetch_data
  name: forecast_line_plot
  name: intermittent_forecast
  name: list_models
  name: hdi_artifacts
  name: ts_dataset_report
  name: ts_check
  name: ts_outlier_detection
  name: classification_tool
  name: regression_tool
  name: ts_make_future_table
  name: SelectStatement_to_table
  name: massive_automatic_timeseries_fit_and_save
  name: massive_automatic_timeseries_load_model_and_predict
  name: massive_automatic_timeseries_load_model_and_score
  name: massive_ts_check
  name: ts_make_future_table_for_massive_forecast
  name: massive_ts_outlier_detection
🔍 calling fetch_data...


In [16]:
print(result.content[0].text)

| Date      |   Confirmed |   Recovered |   Deaths |   Increase rate |
|:----------|------------:|------------:|---------:|----------------:|
| 1/22/2020 |         555 |          28 |       17 |        nan      |
| 1/23/2020 |         654 |          30 |       18 |         17.8378 |
| 1/24/2020 |         941 |          36 |       26 |         43.8838 |
| 1/25/2020 |        1434 |          39 |       42 |         52.3911 |
| 1/26/2020 |        2118 |          52 |       56 |         47.6987 |
